# CRF Resume NER — Section Boundary Detection
Tags resume tokens as `B-EXP`/`I-EXP`, `B-EDU`/`I-EDU`, or `O`.

**Fixes applied (v2):**
- Inference tokeniser now mirrors training tokenisation exactly
- Stray-label guard cell added before training
- Expanded grid search: `c1 ∈ {0.01, 0.1, 1.0}` × `c2 ∈ {0.001, 0.01, 0.1}`
- Raw `word` feature removed; only `word.lower()` kept to reduce overfitting
- Entity-only F1 reporting added (not dominated by O class)


In [ ]:
# ================================================================
#  CONFIGURATION  <- edit FOLDER_PATH before running
# ================================================================
FOLDER_PATH = './data'          # folder containing the 5 JSON files
FILE_NAMES  = [
    'deepseek_json_1.json',
    'deepseek_json_2.json',
    'deepseek_json_3.json',
    'deepseek_json_4.json',
    'deepseek_json_5.json',
]
MODEL_DIR  = './models'
MODEL_PATH = f'{MODEL_DIR}/crf_resume_ner.pkl'
RANDOM_SEED = 42

# ================================================================
#  IMPORTS
# ================================================================
import os, json, re, random, warnings
from itertools import product
from collections import Counter

import joblib
import numpy as np
import sklearn_crfsuite
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(MODEL_DIR, exist_ok=True)
print('Config OK - imports loaded.')


In [ ]:
# ================================================================
#  Load all 5 JSON files and merge into one list of resume dicts.
#  Each entry: {'resume_id': str, 'tokens': [[word, tag], ...]}
# ================================================================
all_resumes = []

for fname in FILE_NAMES:
    fpath = os.path.join(FOLDER_PATH, fname)
    with open(fpath, 'r', encoding='utf-8') as fh:
        records = json.load(fh)
    all_resumes.extend(records)
    print(f'  Loaded {len(records):>3} resumes  <-  {fname}')

print(f'\nTotal resumes loaded: {len(all_resumes)}')

# Sanity check
sample = all_resumes[0]
print(f'\nFirst resume ID : {sample["resume_id"]}')
print(f'Token count     : {len(sample["tokens"])}')
print(f'First 6 tokens  : {sample["tokens"][:6]}')

# Tag distribution
all_tags = [tag for r in all_resumes for _, tag in r['tokens']]
tag_dist = Counter(all_tags)
print('\nTag distribution across all tokens:')
for tag, cnt in sorted(tag_dist.items()):
    pct = cnt / len(all_tags) * 100
    print(f'  {tag:<12}  {cnt:6d}  ({pct:.1f}%)')


In [ ]:
# ================================================================
#  STRAY-LABEL GUARD  (audit fix #2)
#  Print every unique tag found in the data before training.
#  If anything other than the expected 5 appears, investigate
#  the source file before proceeding.
# ================================================================
EXPECTED_LABELS = {'B-EXP', 'I-EXP', 'B-EDU', 'I-EDU', 'O'}

unique_tags = set(tag for r in all_resumes for _, tag in r['tokens'])
print('Unique tags found in data:', sorted(unique_tags))

stray = unique_tags - EXPECTED_LABELS
if stray:
    print(f'\nWARNING: unexpected labels found: {stray}')
    print('Locate and fix these in your JSON files before training.')
else:
    print('\nAll labels are valid - OK to proceed.')

# Also scan for any slash-containing tokens to understand tokenisation
print('\nSlash-containing tokens in training data (sample):')
seen = set()
for r in all_resumes:
    for w, _ in r['tokens']:
        if '/' in w and w not in seen:
            seen.add(w)
for tok in sorted(seen)[:40]:
    print(f'  {tok!r}')
if len(seen) > 40:
    print(f'  ... and {len(seen)-40} more')


In [ ]:
# ================================================================
#  CANONICAL TOKENISER  (audit fix #1)
#
#  This function mirrors EXACTLY how the JSON annotation files
#  were created.  It is the single source of truth for splitting;
#  both the training pipeline and the inference cell use it.
#
#  Rules (derived from empirical scan of the annotation data):
#    - Split X/Y -> X, /, Y  for most tokens
#    - DO NOT split:
#        * A/L, O/L        (Sri Lankan academic shorthands)
#        * W/L             (Win/Loss ratio kept as-is in data)
#        * Numeric ratios  e.g. 2021/2022, 3.68/4.00
#        * URLs            https://... or www. ...
#    - CI/CD, TCP/IP, I/O ARE split in the training data,
#      so they are NOT in INSEPARABLE here.
# ================================================================

# Only terms empirically confirmed as unsplit in training data
INSEPARABLE    = {'A/L', 'O/L', 'W/L', 'A/B'}
_URL_RE        = re.compile(r'https?://', re.I)
_PARTIAL_URL   = re.compile(r'^(www\.|github\.com|linkedin\.com|hackerrank\.com)', re.I)
_NUMERIC_RATIO = re.compile(r'^[\d.,]+/[\d.,]+\.?$')


def _split_slash_token(token):
    """
    Given a single word-piece, return a list of sub-tokens.
    Slashes are split into separate tokens unless the token is
    in INSEPARABLE, is a URL, or is a numeric ratio.
    """
    if '/' not in token or token == '/':
        return [token]
    if _URL_RE.search(token) or _PARTIAL_URL.search(token):
        return [token]
    if token in INSEPARABLE:
        return [token]
    if _NUMERIC_RATIO.match(token):
        return [token]
    # Split on every slash
    parts = token.split('/')
    result = []
    for i, p in enumerate(parts):
        if p:
            result.append(p)
        if i < len(parts) - 1:
            result.append('/')
    return result if result else [token]


def canonical_tokenise(text):
    """
    Tokenise raw resume text to match the training tokenisation.
    Steps:
      1. Split on whitespace.
      2. Peel leading / trailing punctuation into separate tokens.
      3. Apply slash-splitting rules.
    Returns a list of token strings.
    """
    tokens = []
    for raw in text.split():
        # Peel leading punctuation
        m = re.match(r'^([^\w]+)(.*)', raw)
        if m:
            tokens.append(m.group(1))
            raw = m.group(2)
        if not raw:
            continue
        # Peel trailing punctuation
        m2    = re.match(r'^(.*\w)([^\w]+)$', raw)
        core  = m2.group(1) if m2 else raw
        trail = m2.group(2) if m2 else ''

        tokens.extend(_split_slash_token(core))
        if trail:
            tokens.append(trail)

    return [t for t in tokens if t.strip()]


print('canonical_tokenise() defined.')

# ---- Smoke-test to verify CI/CD is NOW split correctly ----
demo_text = 'Experience with CI/CD pipelines and A/L Maths'
print(f'\nTest tokenisation:')
print(f'  Input : {demo_text!r}')
print(f'  Output: {canonical_tokenise(demo_text)}')
# Expected: ['Experience', 'with', 'CI', '/', 'CD', 'pipelines', 'and', 'A/L', 'Maths']


In [ ]:
# ================================================================
#  Word shape helper
#  Replaces: uppercase->X, lowercase->x, digit->d, keeps rest.
#  e.g. 'Python3' -> 'Xxxxxx0'  /  'BSc' -> 'XXx'
# ================================================================
def word_shape(w):
    out = []
    for ch in w:
        if ch.isupper():   out.append('X')
        elif ch.islower(): out.append('x')
        elif ch.isdigit(): out.append('d')
        else:              out.append(ch)
    return ''.join(out)


# ================================================================
#  Build feature dict for one token at position i.
#  NOTE: raw 'word' feature removed (audit fix #4) to reduce
#  overfitting on rare capitalisations with small data.
#  Only word.lower() is kept as the lexical anchor.
# ================================================================
def token_features(words, i):
    w = words[i]

    feats = {
        # Current token (lowercase only - no raw form to reduce overfit)
        'word.lower()':   w.lower(),
        'word.isupper()': w.isupper(),
        'word.istitle()': w.istitle(),
        'word.isdigit()': w.isdigit(),
        'word.shape':     word_shape(w),

        # Prefixes
        'prefix-1': w[:1],
        'prefix-2': w[:2],
        'prefix-3': w[:3],
        'prefix-4': w[:4],

        # Suffixes
        'suffix-1': w[-1:],
        'suffix-2': w[-2:],
        'suffix-3': w[-3:],
        'suffix-4': w[-4:],

        # Position flags
        'BOS': (i == 0),
        'EOS': (i == len(words) - 1),
    }

    # Previous token context
    if i > 0:
        pw = words[i - 1]
        feats['prev_word']  = pw.lower()
        feats['prev_shape'] = word_shape(pw)
    else:
        feats['prev_word']  = 'BOS'
        feats['prev_shape'] = 'BOS'

    # Next token context
    if i < len(words) - 1:
        nw = words[i + 1]
        feats['next_word']  = nw.lower()
        feats['next_shape'] = word_shape(nw)
    else:
        feats['next_word']  = 'EOS'
        feats['next_shape'] = 'EOS'

    return feats


# ================================================================
#  Convert one resume's [[word, tag], ...] into (X_seq, y_seq)
# ================================================================
def resume_to_Xy(token_tag_pairs):
    words = [w for w, _ in token_tag_pairs]
    tags  = [t for _, t in token_tag_pairs]
    X_seq = [token_features(words, i) for i in range(len(words))]
    return X_seq, tags


print('Feature functions ready.')

# Smoke-test
demo = ['Education', 'BSc', 'Computer', 'Science', '2021']
print('\nSample features for token "BSc" (i=1):')
for k, v in token_features(demo, 1).items():
    print(f'  {k:<22} : {v}')


In [ ]:
# ================================================================
#  Resume-level split: 80% train / 10% val / 10% test
#  Shuffled by resume so no token leakage across splits.
# ================================================================
shuffled = all_resumes.copy()
random.shuffle(shuffled)

n       = len(shuffled)
n_test  = max(1, round(n * 0.10))
n_val   = max(1, round(n * 0.10))
n_train = n - n_val - n_test

train_resumes = shuffled[:n_train]
val_resumes   = shuffled[n_train : n_train + n_val]
test_resumes  = shuffled[n_train + n_val:]

# Verify no overlap
train_ids = {r['resume_id'] for r in train_resumes}
val_ids   = {r['resume_id'] for r in val_resumes}
test_ids  = {r['resume_id'] for r in test_resumes}
assert not (train_ids & val_ids),  'Train/Val overlap!'
assert not (train_ids & test_ids), 'Train/Test overlap!'
assert not (val_ids   & test_ids), 'Val/Test overlap!'

print(f'Total resumes : {n}')
print(f'  Train       : {len(train_resumes)}')
print(f'  Val         : {len(val_resumes)}')
print(f'  Test        : {len(test_resumes)}')
print('No split overlap confirmed.')


def build_Xy(resume_list):
    X, y = [], []
    for r in resume_list:
        xs, ys = resume_to_Xy(r['tokens'])
        X.append(xs)
        y.append(ys)
    return X, y


X_train, y_train = build_Xy(train_resumes)
X_val,   y_val   = build_Xy(val_resumes)
X_test,  y_test  = build_Xy(test_resumes)

# Combined train+val for final retraining after grid search
X_trainval = X_train + X_val
y_trainval = y_train + y_val

print(f'\nTrain  : {len(X_train)} seqs  ({sum(len(s) for s in X_train):,} tokens)')
print(f'Val    : {len(X_val)} seqs  ({sum(len(s) for s in X_val):,} tokens)')
print(f'Test   : {len(X_test)} seqs  ({sum(len(s) for s in X_test):,} tokens)')


In [ ]:
# ================================================================
#  Grid search over c1 (L1) x c2 (L2) regularisation.
#  Expanded grid (audit fix #3): 3x3 instead of 2x2.
#  Scored by macro F1 on entity labels only (B/I-EXP, B/I-EDU).
# ================================================================
C1_VALUES   = [0.01, 0.1, 1.0]          # expanded from [0.1, 1.0]
C2_VALUES   = [0.001, 0.01, 0.1]        # expanded from [0.01, 0.1]
ENTITY_LBLS = ['B-EXP', 'I-EXP', 'B-EDU', 'I-EDU']

grid_log    = []
best_score  = -1.0
best_params = None

print(f'Grid search  c1={C1_VALUES}  x  c2={C2_VALUES}')
print(f'Metric: macro F1 on entity labels (validation set)\n')
print(f'{"c1":>6}  {"c2":>7}  {"val_F1":>8}')
print('-' * 30)

for c1, c2 in product(C1_VALUES, C2_VALUES):
    crf = sklearn_crfsuite.CRF(
        algorithm='lbfgs',
        c1=c1, c2=c2,
        max_iterations=200,
        all_possible_transitions=True,
    )
    crf.fit(X_train, y_train)

    y_val_pred = crf.predict(X_val)
    y_vt = [t for seq in y_val      for t in seq]
    y_vp = [t for seq in y_val_pred for t in seq]

    present = [l for l in ENTITY_LBLS if l in y_vt or l in y_vp]
    score   = (f1_score(y_vt, y_vp, labels=present,
                        average='macro', zero_division=0)
               if present else 0.0)

    grid_log.append({'c1': c1, 'c2': c2, 'val_f1': score})
    print(f'{c1:>6}  {c2:>7}  {score:>8.4f}')

    if score > best_score:
        best_score  = score
        best_params = {'c1': c1, 'c2': c2}

print(f'\nBest -> {best_params}   val macro-F1 = {best_score:.4f}')


In [ ]:
# ================================================================
#  Retrain final model on train+val combined using best params.
# ================================================================
final_crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=best_params['c1'],
    c2=best_params['c2'],
    max_iterations=200,
    all_possible_transitions=True,
)
final_crf.fit(X_trainval, y_trainval)

print(f'Final model trained on {len(X_trainval)} sequences '
      f'({sum(len(s) for s in X_trainval):,} tokens)')
print(f'Params: c1={best_params["c1"]}  c2={best_params["c2"]}')


In [ ]:
# ================================================================
#  Evaluate final model on the held-out test set.
#  Reports both full classification report AND entity-only F1
#  (audit fix #3 - avoid metric dominated by O class).
# ================================================================
y_test_pred = final_crf.predict(X_test)

y_true_flat = [t for seq in y_test      for t in seq]
y_pred_flat = [t for seq in y_test_pred for t in seq]

ALL_LABELS    = ['B-EXP', 'I-EXP', 'B-EDU', 'I-EDU', 'O']
ENTITY_LABELS = ['B-EXP', 'I-EXP', 'B-EDU', 'I-EDU']

print('=' * 60)
print('      EVALUATION REPORT  (test set)')
print('=' * 60)
print(classification_report(
    y_true_flat, y_pred_flat,
    labels=ALL_LABELS, zero_division=0
))

acc = accuracy_score(y_true_flat, y_pred_flat)
wf1 = f1_score(y_true_flat, y_pred_flat,
               average='weighted', zero_division=0)

# Entity-only macro F1 (not skewed by O class)
present_ents = [l for l in ENTITY_LABELS
                if l in y_true_flat or l in y_pred_flat]
entity_f1 = f1_score(y_true_flat, y_pred_flat,
                     labels=present_ents,
                     average='macro', zero_division=0)

print(f'Token-level accuracy        : {acc:.4f}')
print(f'Weighted avg F1 (all labels): {wf1:.4f}')
print(f'Entity-only macro F1        : {entity_f1:.4f}  <-- key metric')


In [ ]:
# ================================================================
#  Confusion matrix heatmap, saved to models/
# ================================================================
ALL_LABELS = ['B-EXP', 'I-EXP', 'B-EDU', 'I-EDU', 'O']
cm = confusion_matrix(y_true_flat, y_pred_flat, labels=ALL_LABELS)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=ALL_LABELS, yticklabels=ALL_LABELS,
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title('Confusion Matrix - CRF Resume NER (test set)', fontsize=12)
plt.tight_layout()
cm_path = os.path.join(MODEL_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Saved -> {cm_path}')


In [ ]:
# ================================================================
#  Print gold vs predicted for the first 3 test resumes.
# ================================================================
SHOW_N_RESUMES = 3
SHOW_N_TOKENS  = 20

for resume, y_true_seq, y_pred_seq in zip(
        test_resumes[:SHOW_N_RESUMES], y_test, y_test_pred):

    rid   = resume['resume_id']
    words = [w for w, _ in resume['tokens']]

    print(f'\n{"="*58}')
    print(f'Resume : {rid}')
    print(f'{"TOKEN":<22}  {"GOLD":<10}  {"PRED":<10}  OK?')
    print('-' * 52)

    for w, gold, pred in zip(
            words[:SHOW_N_TOKENS],
            y_true_seq[:SHOW_N_TOKENS],
            y_pred_seq[:SHOW_N_TOKENS]):
        mark = 'v' if gold == pred else 'X'
        print(f'  {w:<20}  {gold:<10}  {pred:<10}  {mark}')

    extra = len(words) - SHOW_N_TOKENS
    if extra > 0:
        print(f'  ... ({extra} more tokens not shown)')


In [ ]:
# ================================================================
#  Save trained model with joblib.
# ================================================================
joblib.dump(final_crf, MODEL_PATH)

print(f'Model saved  ->  {MODEL_PATH}')
print(f'Best params  :   c1={best_params["c1"]}  c2={best_params["c2"]}')
print(f'Val macro-F1 :   {best_score:.4f}')
print(f'Test wt-F1   :   {wf1:.4f}')
print(f'Entity F1    :   {entity_f1:.4f}')


## Inference on New Raw Text
Uses **`canonical_tokenise()`** defined above — identical tokenisation to training.
CI/CD will now be split into `CI`, `/`, `CD` exactly as the model learned.

In [ ]:
# ================================================================
#  INFERENCE  (audit fix #1 applied)
#  Uses canonical_tokenise() - same rules as training data.
#  CI/CD -> CI, /, CD   A/L stays as A/L   2021/2022 stays intact
# ================================================================

def predict_resume(text, model):
    """Tokenise raw text and return [(word, predicted_tag), ...]."""
    words  = canonical_tokenise(text)
    X_seq  = [token_features(words, i) for i in range(len(words))]
    y_pred = model.predict([X_seq])[0]
    return list(zip(words, y_pred))


# Load model and run on a sample resume snippet
loaded_model = joblib.load(MODEL_PATH)
print(f'Model loaded from: {MODEL_PATH}\n')

# Note: CI/CD will be tokenised as CI, /, CD (matches training)
sample_cv = """
Education
BSc Computer Science University of Moratuwa 2021
A/L Mathematics Stream 2017

Experience
Software Engineer Intern Virtusa Colombo Jan 2023 Jun 2023
Developed REST APIs using Python and FastAPI Docker CI/CD

Skills
Python Java React Node.js PostgreSQL Docker
"""

results = predict_resume(sample_cv, loaded_model)

print(f'{"TOKEN":<28}  PREDICTED LABEL')
print('-' * 50)
for word, tag in results:
    print(f'  {word:<26}  {tag}')


In [ ]:
# ================================================================
#  Visualise expanded grid-search results as a heatmap.
# ================================================================
import pandas as pd

df_grid = pd.DataFrame(grid_log)
pivot   = df_grid.pivot(index='c1', columns='c2', values='val_f1')

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='YlGnBu',
    linewidths=0.5, ax=ax
)
ax.set_title('Grid Search - Validation Macro F1 (entity labels)', fontsize=11)
ax.set_xlabel('c2  (L2 regularisation)')
ax.set_ylabel('c1  (L1 regularisation)')
plt.tight_layout()
gs_path = os.path.join(MODEL_DIR, 'grid_search_heatmap.png')
plt.savefig(gs_path, dpi=150)
plt.show()
print(f'Saved -> {gs_path}')
